In [ ]:
## Next Word Prediction Using LSTM 

In [1]:
import pandas as pd

In [2]:
data = pd.read_json('News_Category_Dataset_v3.json', lines=True)

In [3]:
data.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  str           
 1   headline           209527 non-null  str           
 2   category           209527 non-null  str           
 3   short_description  209527 non-null  str           
 4   authors            209527 non-null  str           
 5   date               209527 non-null  datetime64[us]
dtypes: datetime64[us](1), str(5)
memory usage: 71.2 MB


In [ ]:
data.isna().sum()

In [5]:
data.duplicated().sum()

np.int64(13)

In [6]:
data.drop_duplicates(inplace=True)

In [7]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.authors.value_counts()

In [ ]:
data.authors.nunique()

In [ ]:
data.category.value_counts()

In [ ]:
data.category.nunique()

In [8]:
data.category = data.category.astype('category')

In [12]:
data.info()

<class 'pandas.DataFrame'>
Index: 209514 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209514 non-null  str           
 1   headline           209514 non-null  str           
 2   category           209514 non-null  category      
 3   short_description  209514 non-null  str           
 4   authors            209514 non-null  str           
 5   date               209514 non-null  datetime64[us]
dtypes: category(1), datetime64[us](1), str(4)
memory usage: 69.7 MB


In [16]:
data = data.short_description.head(20000)

In [17]:
data

0        Health experts said it is too early to predict...
1        He was subdued by passengers and crew when he ...
2        "Until you have a dog you don't understand wha...
3        "Accidentally put grown-up toothpaste on my to...
4        Amy Cooper accused investment firm Franklin Te...
                               ...                        
19995    He and his wife Grace would be granted full im...
19996    Transition is the name of the game in the Midd...
19997    The breakdown of coalition talks has plunged G...
19998    In a major, thorough and truly important piece...
19999    In recent weeks, Saudi Arabia has become more ...
Name: short_description, Length: 20000, dtype: str

In [18]:
data.info()

<class 'pandas.Series'>
Index: 20000 entries, 0 to 19999
Series name: short_description
Non-Null Count  Dtype
--------------  -----
20000 non-null  str  
dtypes: str(1)
memory usage: 2.1 MB


In [19]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np
from keras.utils import pad_sequences
from keras.layers import Embedding, LSTM, Dense
from keras.models import Sequential

In [20]:
tokenizer = Tokenizer(oov_token='Nothing')

In [21]:
tokenizer.fit_on_texts(data)

In [22]:
tokenizer.word_index

{'Nothing': 1,
 'the': 2,
 'to': 3,
 'a': 4,
 'of': 5,
 'and': 6,
 'in': 7,
 'for': 8,
 'is': 9,
 'on': 10,
 'said': 11,
 'that': 12,
 'was': 13,
 'with': 14,
 'his': 15,
 'it': 16,
 'at': 17,
 'he': 18,
 'are': 19,
 'as': 20,
 'has': 21,
 'have': 22,
 'an': 23,
 'from': 24,
 'be': 25,
 'this': 26,
 'president': 27,
 'trump': 28,
 '”': 29,
 'about': 30,
 'by': 31,
 'her': 32,
 'new': 33,
 'not': 34,
 'but': 35,
 'after': 36,
 'you': 37,
 'who': 38,
 'i': 39,
 'will': 40,
 'one': 41,
 'people': 42,
 'more': 43,
 'their': 44,
 'been': 45,
 'year': 46,
 'out': 47,
 'former': 48,
 'they': 49,
 'up': 50,
 'she': 51,
 'u': 52,
 'were': 53,
 's': 54,
 'all': 55,
 'we': 56,
 'just': 57,
 'what': 58,
 'first': 59,
 'over': 60,
 'had': 61,
 'house': 62,
 'than': 63,
 'says': 64,
 'when': 65,
 'two': 66,
 'no': 67,
 'would': 68,
 'or': 69,
 'some': 70,
 'like': 71,
 'state': 72,
 "it's": 73,
 'into': 74,
 'show': 75,
 'if': 76,
 'last': 77,
 'also': 78,
 'time': 79,
 'say': 80,
 'your': 81,
 'cou

In [23]:
len(tokenizer.word_index)

26343

In [24]:
tokenizer.document_count

20000

In [25]:
data = tokenizer.texts_to_sequences(data)

In [26]:
data

[[163,
  352,
  11,
  16,
  9,
  158,
  537,
  3,
  5038,
  348,
  1517,
  68,
  2403,
  50,
  14,
  2,
  13529,
  198,
  4129,
  5,
  2,
  33,
  13530,
  2,
  52,
  54,
  1300,
  8,
  2,
  897],
 [18,
  13,
  13531,
  31,
  2078,
  6,
  2545,
  65,
  18,
  2175,
  3,
  2,
  107,
  5,
  2,
  3045,
  36,
  2,
  4130,
  232,
  3,
  2,
  52,
  54,
  9681,
  247,
  7,
  973,
  1037],
 [507, 37, 22, 4, 1342, 37, 192, 1518, 58, 82, 25, 7741],
 [4131,
  258,
  3543,
  50,
  9682,
  10,
  93,
  13532,
  13533,
  6,
  18,
  9683,
  71,
  39,
  13,
  3833,
  15,
  3285,
  14,
  4,
  1065,
  13534,
  9684,
  7,
  13535,
  7742],
 [2079,
  2714,
  133,
  4509,
  1519,
  5039,
  13536,
  5,
  7743,
  2546,
  32,
  6,
  9685,
  32,
  4,
  749,
  36,
  197,
  5,
  2,
  1264,
  803,
  2404,
  338,
  785],
 [2,
  3834,
  46,
  94,
  191,
  13,
  592,
  402,
  17,
  2,
  403,
  1065,
  1520,
  10,
  640,
  51,
  13,
  199,
  418,
  307,
  36,
  32,
  186,
  353,
  32,
  898,
  394,
  11],
 [1185,
  12,


In [27]:
input_sequences = []

for seq in data:
    if len(seq) < 2:
        continue
    for i in range(1, len(seq)):
        ngram = seq[:i+1]
        input_sequences.append(ngram)


In [28]:
len(input_sequences)

300530

In [29]:
input_sequences

[[163, 352],
 [163, 352, 11],
 [163, 352, 11, 16],
 [163, 352, 11, 16, 9],
 [163, 352, 11, 16, 9, 158],
 [163, 352, 11, 16, 9, 158, 537],
 [163, 352, 11, 16, 9, 158, 537, 3],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038, 348],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038, 348, 1517],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038, 348, 1517, 68],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038, 348, 1517, 68, 2403],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038, 348, 1517, 68, 2403, 50],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038, 348, 1517, 68, 2403, 50, 14],
 [163, 352, 11, 16, 9, 158, 537, 3, 5038, 348, 1517, 68, 2403, 50, 14, 2],
 [163,
  352,
  11,
  16,
  9,
  158,
  537,
  3,
  5038,
  348,
  1517,
  68,
  2403,
  50,
  14,
  2,
  13529],
 [163,
  352,
  11,
  16,
  9,
  158,
  537,
  3,
  5038,
  348,
  1517,
  68,
  2403,
  50,
  14,
  2,
  13529,
  198],
 [163,
  352,
  11,
  16,
  9,
  158,
  537,
  3,
  5038,
  348,
  1517,
  68,
  2403,
  50,
  14,
  2,
  13

In [30]:
max([len(i) for i in input_sequences])

44

In [31]:
input = pad_sequences(input_sequences, padding='pre')

In [32]:
input

array([[   0,    0,    0, ...,    0,  163,  352],
       [   0,    0,    0, ...,  163,  352,   11],
       [   0,    0,    0, ...,  352,   11,   16],
       ...,
       [   0,    0,    0, ...,   10, 3817,  560],
       [   0,    0,    0, ..., 3817,  560,  354],
       [   0,    0,    0, ...,  560,  354,   12]],
      shape=(300530, 44), dtype=int32)

In [33]:
input[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0, 163, 352], dtype=int32)

In [34]:
X = input[:,:-1]
y = input[:, -1]

In [35]:
X[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0, 163], dtype=int32)

In [36]:
y[0]

np.int32(352)

In [37]:
y = np.array(y)

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
X_train.shape

(240424, 43)

In [41]:
vocab_size = len(tokenizer.word_index) + 1
vocab_size

26344

In [42]:
max_len = X_train.shape[1]
max_len

43

In [43]:
model = Sequential(
    [
        Embedding(input_dim=vocab_size,output_dim=128,input_length=max_len),
        LSTM(256),
        Dense(vocab_size, activation='softmax')
    ]
)

c:\Users\ManishAgrahari\Desktop\my_learning\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [44]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [45]:
model.build(input_shape=(None, max_len))

In [46]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 43, 128)        │     3,372,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 26344)          │     6,770,408 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,536,680 (40.19 MB)

 Trainable params: 10,536,680 (40.19 MB)

 Non-trainable params: 0 (0.00 B)

In [49]:
model.fit(X_train, y_train, epochs=1, validation_data=(X_test, y_test), batch_size=256)

940/940 ━━━━━━━━━━━━━━━━━━━━ 463s 493ms/step - accuracy: 0.0748 - loss: 7.2751 - val_accuracy: 0.0929 - val_loss: 7.0561


In [50]:
model.save('nextword_model.h5')

In [51]:
import pickle

with open('tokenizer.pkl', 'wb') as file:
    pickle.dump(tokenizer, file)